# 실습 2: 콘텐츠 조정 (Content Moderation)

## 학습 목표
- Amazon Rekognition의 `detect_moderation_labels` API로 부적절한 콘텐츠를 감지합니다.
- 계층적 카테고리 구조(ParentName → Name)를 이해합니다.

## API 개요
`detect_moderation_labels`는 성인 콘텐츠, 폭력, 혐오 등 부적절한 이미지를 감지합니다.
- **MinConfidence**: 결과에 포함할 최소 신뢰도 (%)
- **반환값**: `response['ModerationLabels']`
  - `Name`: 세부 카테고리 (예: 'Explicit Nudity')
  - `ParentName`: 상위 카테고리 (예: 'Explicit Content')
  - `Confidence`: 신뢰도


In [1]:
import boto3
import os
from PIL import Image
import matplotlib.pyplot as plt

# ✅ [제공 코드]
rekognition = boto3.client('rekognition', region_name='ap-northeast-2')
IMAGE_DIR = './images/'
image_filename = 'lab02.jpg'
image_path = os.path.join(IMAGE_DIR, image_filename)

def load_image_bytes(path):
    with open(path, 'rb') as f:
        return f.read()

image_bytes = load_image_bytes(image_path)
print(f"이미지 경로: {image_path}")
print(f"이미지 크기: {len(image_bytes):,} bytes")

이미지 경로: ./images/lab02.jpg
이미지 크기: 127,940 bytes


## ✏️ TODO 1: detect_moderation_labels API 호출

아래 빈칸을 채워 콘텐츠 조정 API를 호출하세요.


In [2]:
# ✏️ TODO 1: API 호출 코드를 완성하세요
response = rekognition.detect_moderation_labels( # ← detect_moderation_labels
    Image={'Bytes': image_bytes},  # ← image_bytes
    MinConfidence=50      # ← 50
)

labels = response['ModerationLabels']
print(f"감지된 부적절 콘텐츠: {len(labels)}건")

감지된 부적절 콘텐츠: 3건


## ✏️ TODO 2: 카테고리별 결과 출력

감지된 레이블을 상위 카테고리(ParentName)와 세부 항목(Name)으로 구분하여 출력하세요.


In [3]:
# ✏️ TODO 2: 결과를 계층 구조로 출력하세요
if len(labels) == 0:
    print("✅ 부적절한 콘텐츠가 감지되지 않았습니다.")
else:
    print("⚠️ 부적절한 콘텐츠가 감지되었습니다:")
    print("-" * 50)
    for label in labels:
        parent = label.get('ParentName', '(최상위)') # ← 'ParentName'
        name   = label['Name']                # ← 'Name'
        conf   = label['Confidence']                # ← 'Confidence'
        print(f"  [{parent}] > {name}: {conf:.1f}%")

⚠️ 부적절한 콘텐츠가 감지되었습니다:
--------------------------------------------------
  [Drugs & Tobacco Paraphernalia & Use] > Smoking: 93.4%
  [Drugs & Tobacco] > Drugs & Tobacco Paraphernalia & Use: 93.4%
  [] > Drugs & Tobacco: 93.4%


## ✏️ TODO 3: 안전 여부 판정 로직

Confidence가 80% 이상인 레이블이 있으면 '위험', 없으면 '안전'으로 출력하는 로직을 작성하세요.


In [5]:
# ✏️ TODO 3: 안전 여부 판정 로직을 완성하세요
threshold = 80.0  # ← 80.0

# 조건: threshold 이상인 레이블이 하나라도 있으면 위험
is_unsafe = any(label['Confidence'] >= threshold for label in labels)

if is_unsafe:
    print(f"🚫 이미지 판정: 위험 (신뢰도 {threshold}% 이상 콘텐츠 감지)")
else:
    print(f"✅ 이미지 판정: 안전")

🚫 이미지 판정: 위험 (신뢰도 80.0% 이상 콘텐츠 감지)


## 💡 심화 도전
1. `MinConfidence`를 30으로 낮추면 결과가 어떻게 달라지나요?
2. 이미지가 '안전'하다고 판정된 경우에도 ModerationLabels가 반환될 수 있을까요?
3. 여러 이미지를 반복 처리하여 안전/위험 이미지를 자동 분류해보세요.
